# 11 - Segmentación territorial y validación

- Unidad: barrio (48 observaciones).
- K-means y agrupamiento jerárquico se comparan sobre componentes principales retenidos.
- Se evalúan silueta, estabilidad ante submuestras e interpretación sustantiva.
- La segmentación es exploratoria; no define categorías oficiales ni prioriza por sí sola una política.
- Insumos: coordenadas PCA de 10 y KPIs por barrio de 07.
- Salidas: etiquetas y perfiles en `data/processed/` y `data/processed/reports/`.

## 1. Preparación y selección de dimensiones

- Se retienen los componentes que explican al menos 80% de la varianza, con máximo de cinco.
- La regla mantiene parsimonia con 48 barrios.
- Se comprueba que haya casos suficientes para evaluar los valores candidatos de k.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
from src import config

REPORTS = config.REPORTS_DIR
scores = pd.read_csv(config.PROCESSED_DIR / "pe3_pca_barrios_scores_v1.csv")
varianza = pd.read_csv(REPORTS / "pe3_pca_varianza_v1.csv")
kb = pd.read_csv(config.PROCESSED_DIR / "kpis_barrio_v1.csv")

pc_cols = [c for c in scores.columns if c.startswith("PC") and c[2:].isdigit()]
pc_cols = sorted(pc_cols, key=lambda c: int(c[2:]))
n_pc_80 = int(np.searchsorted(varianza["varianza_explicada"].cumsum().to_numpy(), .80) + 1)
n_pc = min(max(n_pc_80, 2), 5, len(pc_cols))
X = scores[pc_cols[:n_pc]].to_numpy()
barrios = scores["barrio_oficial"].astype(str)
print("Componentes usados:", pc_cols[:n_pc])
print("Varianza acumulada:", round(varianza["varianza_explicada"].iloc[:n_pc].sum(), 3))

## 2. Comparación de soluciones

- Se comparan k entre 2 y 5.
- La silueta evalúa separación geométrica, no validez social o habitacional.
- La estabilidad se estima con submuestras aleatorias; ARI compara asignaciones de los barrios compartidos.
- Se informa un candidato mecánico y se revisa su interpretación antes de elegir la solución final.

In [ ]:
rng = np.random.default_rng(2026)
resultados = []
modelos = {}
n = len(X)
for k in range(2, min(5, n - 1) + 1):
    km = KMeans(n_clusters=k, n_init=50, random_state=2026).fit(X)
    modelos[k] = km
    sil_km = silhouette_score(X, km.labels_)
    ward = AgglomerativeClustering(n_clusters=k, linkage="ward").fit(X)
    sil_ward = silhouette_score(X, ward.labels_)

    soluciones = []
    for rep in range(30):
        idx = np.sort(rng.choice(n, size=max(k + 2, int(.8 * n)), replace=False))
        labels = KMeans(n_clusters=k, n_init=20, random_state=rep).fit_predict(X[idx])
        soluciones.append((idx, labels))
    ari = []
    for i in range(len(soluciones)):
        idx_i, lab_i = soluciones[i]
        map_i = dict(zip(idx_i, lab_i))
        for j in range(i + 1, len(soluciones)):
            idx_j, lab_j = soluciones[j]
            comunes = np.intersect1d(idx_i, idx_j)
            if len(comunes) > k:
                ari.append(adjusted_rand_score(
                    [map_i[q] for q in comunes],
                    [dict(zip(idx_j, lab_j))[q] for q in comunes]
                ))
    resultados.append({
        "k": k, "silueta_kmeans": sil_km, "silueta_ward": sil_ward,
        "ari_mediana_submuestras": float(np.median(ari)) if ari else np.nan
    })

validacion = pd.DataFrame(resultados).sort_values("k")
display(validacion.round(3))
k_candidato = int(validacion.loc[validacion["silueta_kmeans"].idxmax(), "k"])
print("Candidato por silueta K-means (requiere revisión interpretativa):", k_candidato)
validacion.to_csv(REPORTS / "pe3_validacion_clustering_v1.csv", index=False)

## 3. Perfiles y lectura territorial

- Se describe cada grupo con medianas y rangos de los KPIs originales.
- Se muestra la distribución en PC1-PC2; cada punto representa un barrio.
- Los rótulos comunales ayudan a relacionar perfiles con los patrones de 08, sin reemplazar el diagnóstico.

In [ ]:
labels = modelos[k_candidato].labels_
asignacion = pd.DataFrame({"barrio_oficial": barrios, "cluster": labels + 1})
perfil = asignacion.merge(kb, on="barrio_oficial", validate="one_to_one")
columnas_perfil = [
    "airbnb_por_km2", "mediana_precio_m2_venta_usd",
    "mediana_precio_m2_alquiler_usd", "densidad_culturales_km2",
    "densidad_subte_km2", "densidad_educacion_km2", "pct_superficie_verde"
]
columnas_perfil = [c for c in columnas_perfil if c in perfil.columns]
resumen = perfil.groupby("cluster")[columnas_perfil].median().round(2)
resumen.insert(0, "barrios", perfil.groupby("cluster").size())
display(resumen)
display(asignacion.sort_values(["cluster", "barrio_oficial"]))
asignacion.to_csv(config.PROCESSED_DIR / "pe3_clusters_barrios_v1.csv", index=False)
resumen.to_csv(REPORTS / "pe3_perfiles_clusters_v1.csv")

fig, ax = plt.subplots(figsize=(7, 5))
for cluster, grupo in asignacion.groupby("cluster"):
    ix = grupo.index
    ax.scatter(scores.loc[ix, "PC1"], scores.loc[ix, "PC2"], s=45, label="Grupo " + str(cluster))
    for i in ix:
        ax.annotate(barrios.iloc[i], (scores.loc[i, "PC1"], scores.loc[i, "PC2"]),
                    xytext=(3, 2), textcoords="offset points", fontsize=6)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Segmentación exploratoria de barrios")
ax.legend(frameon=False, ncol=2)
plt.tight_layout()
plt.show()

## 4. Decisión metodológica provisional

- Elegir k requiere considerar silueta, estabilidad y si los perfiles tienen sentido para el Instituto.
- Si una solución es inestable o los grupos solo reflejan densidad de publicaciones, no se conserva como segmentación final.
- Los resultados sirven para organizar el análisis posterior; no son una clasificación del déficit habitacional.
- Revisar sensibilidad a variables, número de componentes, método jerárquico y exclusión de barrios con poca cobertura.